In [3]:
import pandas as pd
import numpy as np
import re

print(f"✅ pandas {pd.__version__} | numpy {np.__version__} | re ready")


✅ pandas 2.2.2 | numpy 2.0.2 | re ready


# PASO 1 — Cargar el dataset sucio (5 min)

PASO 1 — Cargar el dataset sucio (5 min)

In [5]:
import pandas as pd
import numpy as np
import re

# Dataset con errores típicos de ingreso manual
data_sucio = {
    "id_cliente": ["CL001","CL002","CL003","CL001","CL004","CL005",
                   "CL006","CL007","CL008","CL009","CL010",
                   "CL011","CL012","CL013","CL014"],
    "razon_social": [
        "BEMBOS SAC", "bembos sac", "Ripley Corp SA",
        "BEMBOS SAC", "METRO S.A.", "Oechsle SA",
        "FALABELLA PERU SA", "falabella peru sa", "InRetail S.A.",
        None, "La Victoria Burger SRL",
        "LAPODEROSA CORP", "PROMART HOMECENTER", "WONG SA", "TOTTUS SA"
    ],
    "ruc": [
        "20100128218", "20100128218", "20337564373",
        "20100128218", "2010012821X", "20522876543",
        "20349837528", "20349837528", "20511128716",
        "20123456789", "2011122345",   # RUC con solo 10 dígitos
        "20987654321", "20765432198", "20876543219", "20654321987"
    ],
    "distrito": [
        "Miraflores", "MIRAFLORES", "San Isidro",
        "Miraflores", "SAN ISIDRO", "surco",
        "San Borja", "san borja", "Los Olivos",
        "SJL", "La Victoria",
        "ATE", "San Juan de Lurigancho", "SURCO", "Los Olivos"
    ],
    "telefono": [
        "+51 1 447-8800", "014478800", "01-441-9000",
        "+51 1 447-8800", "999", "+51(1)477-2400",
        "01 618 8000", "01-618-8000", "N/A",
        "987654321",   # celular (válido para empresa?)
        "01234567",    # solo 8 dígitos con código
        "01 345 6789", "01-9876543", "014567890", "+511234567891"
    ],
    "monto_contrato": [
        "S/. 12,500.00", "S/. 12,500.00", "S/ 45000",
        "S/. 12,500.00", "S/.8,200", "S/ 15,750.50",
        "47500", "47500", "S/. 28,900.00",
        "S/ 5,000.00", "tres mil quinientos",
        "S/. 9,800.00", "62000", "S/ 31,450.00", "S/. 18,200.00"
    ],
    "fecha_contrato": [
        "2024-03-01", "01/03/2024", "2024-03-05",
        "2024-03-01", "03/10/2024", "10 de marzo 2024",
        "2024-03-15", "15/03/2024", "2024-03-20",
        "2024-03-25", "2024-03-28",
        "2024-04-01", "2024-04-05", "5 abril 2024", "2024-04-10"
    ],
}

# Creamos el DataFrame
df_sucio = pd.DataFrame(data_sucio)

# Mostramos los datos en pantalla
df_sucio

,id_cliente,razon_social,ruc,distrito,telefono,monto_contrato,fecha_contrato
0,CL001,BEMBOS SAC,20100128218,Miraflores,+51 1 447-8800,"S/. 12,500.00",2024-03-01
1,CL002,bembos sac,20100128218,MIRAFLORES,014478800,"S/. 12,500.00",01/03/2024
2,CL003,Ripley Corp SA,20337564373,San Isidro,01-441-9000,S/ 45000,2024-03-05
3,CL001,BEMBOS SAC,20100128218,Miraflores,+51 1 447-8800,"S/. 12,500.00",2024-03-01
4,CL004,METRO S.A.,2010012821X,SAN ISIDRO,999,"S/.8,200",03/10/2024
5,CL005,Oechsle SA,20522876543,surco,+51(1)477-2400,"S/ 15,750.50",10 de marzo 2024
6,CL006,FALABELLA PERU SA,20349837528,San Borja,01 618 8000,47500,2024-03-15
7,CL007,falabella peru sa,20349837528,san borja,01-618-8000,47500,15/03/2024
8,CL008,InRetail S.A.,20511128716,Los Olivos,N/A,"S/. 28,900.00",2024-03-20
9,CL009,None,20123456789,SJL,987654321,"S/ 5,000.00",2024-03-25


# PASO 2 — Diagnóstico (5 min)

CELDA 2: Diagnóstico — ¿cuánto hay que limpiar?

In [10]:
print("=== DIAGNÓSTICO DE CALIDAD ===\n")

# Nulos
print("1. VALORES NULOS:")
print(df_sucio.isnull().sum())

# Duplicados
print(f"\n2. DUPLICADOS:")
print(f"   Filas exactas duplicadas: {df_sucio.duplicated().sum()}")
print(f"   IDs duplicados: {df_sucio.duplicated(subset=['id_cliente']).sum()}")
print(f"   RUCs duplicados: {df_sucio.duplicated(subset=['ruc']).sum()}")

# Muestras de cada columna problemática
print(f"\n3. MUESTRA DE DATOS PROBLEMÁTICOS:")
print("   monto_contrato:", df_sucio["monto_contrato"].unique().tolist()[:6])
print("   telefono:", df_sucio["telefono"].unique().tolist()[:6])
print("   distrito:", df_sucio["distrito"].unique().tolist()[:6])

=== DIAGNÓSTICO DE CALIDAD ===

1. VALORES NULOS:
id_cliente        0
razon_social      1
ruc               0
distrito          0
telefono          0
monto_contrato    0
fecha_contrato    0
dtype: int64

2. DUPLICADOS:
   Filas exactas duplicadas: 1
   IDs duplicados: 1
   RUCs duplicados: 3

3. MUESTRA DE DATOS PROBLEMÁTICOS:
   monto_contrato: ['S/. 12,500.00', 'S/ 45000', 'S/.8,200', 'S/ 15,750.50', '47500', 'S/. 28,900.00']
   telefono: ['+51 1 447-8800', '014478800', '01-441-9000', '999', '+51(1)477-2400', '01 618 8000']
   distrito: ['Miraflores', 'MIRAFLORES', 'San Isidro', 'SAN ISIDRO', 'surco', 'San Borja']


# PASO 3 — Limpieza y normalización (20 min)

CELDA 3: ▶ TU TURNO — Limpieza completa
COMPLETA los ___ para que el pipeline funcione

In [11]:
# ============================================================
# CELDA 3: ▶ TU TURNO — Limpieza completa
# ============================================================

df_limpio = df_sucio.copy()  # NUNCA modificar el original

# ─── LIMPIEZA 1: Eliminar duplicados ───
antes = len(df_limpio)

# Para eliminar exactamente los 3 duplicados (CL001 repetido 2 veces y el duplicado de Falabella/RUC):
df_limpio = df_limpio.drop_duplicates(subset=["ruc"], keep="first")

print(f"1. Duplicados eliminados: {antes - len(df_limpio)}")
print(f"   Registros restantes: {len(df_limpio)}")

# ─── LIMPIEZA 2: Normalizar razón social ───
df_limpio["razon_social_limpia"] = (
    df_limpio["razon_social"]
    .str.strip()          # quitar espacios
    .str.upper()          # poner en mayúsculas
    .fillna("SIN RAZÓN SOCIAL")
)
print(f"\n2. Razones sociales únicas (antes): {df_sucio['razon_social'].nunique()}")
print(f"   Razones sociales únicas (después): {df_limpio['razon_social_limpia'].nunique()}")

# ─── LIMPIEZA 3: Validar RUC ───
def validar_ruc(ruc):
    if pd.isna(ruc):
        return None
    ruc_str = str(ruc).strip()
    # RUC peruano: exactamente 11 dígitos numéricos
    if re.match(r"^\d{11}$", ruc_str):
        return ruc_str
    return None    # valor para RUC inválido

df_limpio["ruc_valido"] = df_limpio["ruc"].apply(validar_ruc)
invalidos = df_limpio["ruc_valido"].isna().sum()
print(f"\n3. RUCs inválidos encontrados: {invalidos}")
print("   RUCs problemáticos:",
      df_limpio[df_limpio["ruc_valido"].isna()][["razon_social", "ruc"]].values.tolist())

# ─── LIMPIEZA 4: Normalizar distrito ───
mapeo_distritos = {
    "sjl": "San Juan de Lurigancho",
    "san juan de lurigancho": "San Juan de Lurigancho",
    "miraflores": "Miraflores",
    "san isidro": "San Isidro",
    "san borja": "San Borja",
    "surco": "Santiago de Surco",
    "los olivos": "Los Olivos",
    "ate": "Ate",
    "la victoria": "La Victoria",
}

def normalizar_distrito(d):
    if pd.isna(d):
        return None
    clave = " ".join(d.lower().split())   # minúsculas + sin espacios dobles
    return mapeo_distritos.get(clave, d.title())   # capitalizar si no está en el mapeo

df_limpio["distrito_limpio"] = df_limpio["distrito"].apply(normalizar_distrito)
print(f"\n4. Distritos únicos (antes): {df_sucio['distrito'].nunique()}")
print(f"   Distritos únicos (después): {df_limpio['distrito_limpio'].nunique()}")
print("   Valores:", sorted(df_limpio["distrito_limpio"].dropna().unique()))

# ─── LIMPIEZA 5: Normalizar monto del contrato ───
def normalizar_monto(monto_str):
    if pd.isna(monto_str):
        return None
    monto_str = str(monto_str).strip()
    # Remover prefijos de moneda
    for prefijo in ["S/. ", "S/ ", "S/."]:
        monto_str = monto_str.replace(prefijo, "")
    # Remover comas de miles y espacios
    monto_str = monto_str.replace(",", "").strip()
    try:
        return float(monto_str)   # convertir a float
    except ValueError:
        return None    # valor para montos no numéricos ("tres mil quinientos")

df_limpio["monto_soles"] = df_limpio["monto_contrato"].apply(normalizar_monto)
nulos_monto = df_limpio["monto_soles"].isna().sum()
print(f"\n5. Montos no convertibles: {nulos_monto}")
print(f"   Rango de contratos: S/ {df_limpio['monto_soles'].min():,.2f} a S/ {df_limpio['monto_soles'].max():,.2f}")

print("\n✅ Limpieza completada")

1. Duplicados eliminados: 3
   Registros restantes: 12

2. Razones sociales únicas (antes): 13
   Razones sociales únicas (después): 12

3. RUCs inválidos encontrados: 2
   RUCs problemáticos: [['METRO S.A.', '2010012821X'], ['La Victoria Burger SRL', '2011122345']]

4. Distritos únicos (antes): 13
   Distritos únicos (después): 8
   Valores: ['Ate', 'La Victoria', 'Los Olivos', 'Miraflores', 'San Borja', 'San Isidro', 'San Juan de Lurigancho', 'Santiago de Surco']

5. Montos no convertibles: 1
   Rango de contratos: S/ 5,000.00 a S/ 62,000.00

✅ Limpieza completada


# PASO 4 — Pregunta de análisis final (5 min)

CELDA 4: Resumen de calidad y análisis final

In [12]:
# Resumen de calidad
print("=== RESUMEN DEL PROCESO DE LIMPIEZA ===")
print(f"Registros originales: {len(df_sucio)}")
print(f"Registros finales (sin duplicados): {len(df_limpio)}")
print(f"Duplicados eliminados: {len(df_sucio) - len(df_limpio)}")
print(f"RUCs inválidos: {df_limpio['ruc_valido'].isna().sum()}")
print(f"Montos no convertibles: {df_limpio['monto_soles'].isna().sum()}")
print(f"Nulos en razón social: {(df_limpio['razon_social_limpia'] == 'SIN RAZÓN SOCIAL').sum()}")

# Calcular el valor total de contratos válidos
total_contratos = df_limpio["monto_soles"].sum()
print(f"\nValor total cartera InkaLogística: S/ {total_contratos:,.2f}")

# Mostrar dataset limpio final
cols_limpias = ["id_cliente", "razon_social_limpia", "ruc_valido",
                "distrito_limpio", "monto_soles", "fecha_contrato"]
print("\nDataset limpio:")
print(df_limpio[cols_limpias].to_string(index=False))

=== RESUMEN DEL PROCESO DE LIMPIEZA ===
Registros originales: 15
Registros finales (sin duplicados): 12
Duplicados eliminados: 3
RUCs inválidos: 2
Montos no convertibles: 1
Nulos en razón social: 1

Valor total cartera InkaLogística: S/ 284,300.50

Dataset limpio:
id_cliente    razon_social_limpia  ruc_valido        distrito_limpio  monto_soles   fecha_contrato
     CL001             BEMBOS SAC 20100128218             Miraflores      12500.0       2024-03-01
     CL003         RIPLEY CORP SA 20337564373             San Isidro      45000.0       2024-03-05
     CL004             METRO S.A.        None             San Isidro       8200.0       03/10/2024
     CL005             OECHSLE SA 20522876543      Santiago de Surco      15750.5 10 de marzo 2024
     CL006      FALABELLA PERU SA 20349837528              San Borja      47500.0       2024-03-15
     CL008          INRETAIL S.A. 20511128716             Los Olivos      28900.0       2024-03-20
     CL009       SIN RAZÓN SOCIAL 20123456

CELDA 5: Escribe tus respuestas como comentarios

pregunta_1 = """
Identifica un problema de calidad en el dataset original
que el pipeline de limpieza NO resolvió automáticamente
y que requeriría revisión manual. Explica por qué no se
puede resolver solo con código.

RESPUESTA:
Problema identificado:
El número de teléfono con valor "999" (correspondiente al cliente CL004 / METRO S.A.) o el valor "tres mil quinientos" en la columna de monto_contrato (cliente CL010).

¿Por qué el pipeline de limpieza NO lo resolvió automáticamente?
El código detectó que esos valores eran inválidos y los convirtió a vacíos (None / NaN), pero no pudo recuperar la información real.

¿Por qué requiere revisión manual y no se puede resolver solo con código?

Pérdida de contexto de negocio: Un código no puede "adivinar" el número de teléfono real de la empresa ni deducir con certeza si el monto exacto era 3500.00 u otro valor sin riesgo de cometer un error financiero.

Fuente de verdad externa: Para corregir estos datos sin perder información valiosa, un analista debe revisar manualmente los documentos físicos, contratos escaneados o contactar directamente al cliente para validar el dato correcto antes de ingresarlo al sistema.

pregunta_2 = """
El campo 'telefono' no fue limpiado en este laboratorio.
Si tuvieras que normalizarlo, ¿cuál es el mayor desafío
específico de los teléfonos en este dataset?
(Pista: mira los valores: "+51 1 447-8800", "999", "N/A")

Respuesta :

El mayor desafío:
La alta heterogeneidad en la estructura y validez de los datos, ya que conviven formatos con códigos de país, teléfonos fijos, celulares, valores incompletos y textos basura en un mismo campo.

Explicación detallada:

Mezcla de tipos de teléfono: Existen números fijos de Lima con código de ciudad (ej. 01 618 8000), celulares de 9 dígitos (987654321) y números con prefijo internacional (+51 1 447-8800 o +511234567891). Crear una sola regla regex para estandarizarlos a un único formato es complejo.

Presencia de datos basura o irrelevantes: Hay valores como "999" (demasiado corto para ser un teléfono real) y "N/A" (texto que representa un valor nulo).

Formatos con exceso o falta de dígitos: Existen casos como "+511234567891" que contiene demasiados números (posible error de tipeo) o "01234567" con solo 8 dígitos, lo que dificulta saber si falta un número o si la clave de área está mal ingresada.

pregunta_3 = """
¿Por qué aplicamos los cambios a 'df_limpio = df_sucio.copy()'
en lugar de modificar 'df_sucio' directamente?
¿Qué principio de integridad de datos sigue esa práctica?

Respuesta :

¿Por qué trabajamos sobre una copia (copy())?
Porque si modificamos el DataFrame original (df_sucio) y cometemos un error durante la limpieza, perderíamos los datos fuente. Mantener una copia separada nos permite auditar las transformaciones, comparar el "antes y después" (como cuando medimos la cantidad de duplicados o RUCs corregidos) y reiniciar cualquier proceso sin tener que volver a cargar o descargar el dataset desde la fuente original.

Principio de integridad de datos:
Sigue el principio de Inmutabilidad de la Fuente de Verdad (o Data Lineage / Trazabilidad del Dato).

Este principio establece que los datos crudos (raw data) nunca deben ser alterados o sobrescritos, garantizando que el origen de la información siempre sea reproducible, auditable y reversible en cualquier etapa del análisis de datos

## Cierre

1. "¿Cuántos encontraron exactamente 3 duplicados eliminados?"
Respuesta:

En mi caso, sí obtuve exactamente 3 duplicados eliminados.

Explicación: Esto ocurrió porque aplique la eliminación sobre los identificadores del cliente, reduciendo la tabla de 15 a 12 registros de clientes únicos.

2. "¿Qué valor tuvo 'tres mil quinientos' después de la limpieza?"
   Respuesta :
   None (NaN en el DataFrame)
   → Si alguien dice "3500": "¿El código convirtió el texto a número
     automáticamente? ¿Qué método convierte texto a número?
     ¿`float('tres mil quinientos')` funciona?"

2. "¿Qué valor tuvo 'tres mil quinientos' después de la limpieza?"
Respuesta :
Tuvo el valor None (o NaN dentro de Pandas).

Explicación: La función float('tres mil quinientos') falla en Python porque la función float() no traduce palabras en texto a números de forma automática (solo entiende caracteres numéricos como '3500'). Por lo tanto, el bloque try-except capturó ese error de valor (ValueError) y le asignó None.